# Analyse des facteurs de réussite scolaire

## 1. Chargement des données depuis HDFS

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import when, col

'''
- Liste des port -
Jade     : 8020 
Flavien  : 9000
Samuel   :
Aimane   : 
Roissath :
'''

port = "9000"
spark = SparkSession.builder.appName("parsing_student_performance").getOrCreate()
df = spark.read.csv(f"hdfs://localhost:{port}/projet/donnees/StudentPerformanceFactors_propre.csv", header=True, inferSchema=True)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/07/17 14:36:07 WARN Utils: Your hostname, Opa, resolves to a loopback address: 127.0.1.1; using 192.168.1.66 instead (on interface wlp61s0)
26/07/17 14:36:07 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/07/17 14:36:08 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## 2. Encodage des variables catégorielles

In [2]:
df = df.withColumn(
    "Implication_Parentale_num",
    when(df.Implication_Parentale == "Haut", 2)
    .when(df.Implication_Parentale == "Moyen", 1)
    .when(df.Implication_Parentale == "Bas", 0)
)

df = df.withColumn(
    "Influence_Entourage_num",
    when(df.Influence_Entourage == "Negative", -1)
    .when(df.Influence_Entourage == "Positif", 1)
    .when(df.Influence_Entourage == "Neutre", 0)
)

df = df.withColumn(
    "Acces_aux_Ressources_num",
    when(df.Acces_aux_Ressources == "Bas", 0)
    .when(df.Acces_aux_Ressources == "Moyen", 1)
    .when(df.Acces_aux_Ressources == "Haut", 2)
)

df = df.withColumn(
    "Niveau_Motivation_num",
    when(df.Niveau_Motivation == "Bas", 0)
    .when(df.Niveau_Motivation == "Moyen", 1)
    .when(df.Niveau_Motivation == "Haut", 2)
)

df = df.withColumn(
    "Qualite_Enseignant_num",
    when(df.Qualite_Enseignant == "Bas", 0)
    .when(df.Qualite_Enseignant == "Moyen", 1)
    .when(df.Qualite_Enseignant == "Haut", 2)
)

df = df.withColumn(
    "Revenu_Famille_num",
    when(df.Revenu_Famille == "Bas", 0)
    .when(df.Revenu_Famille == "Moyen", 1)
    .when(df.Revenu_Famille == "Haut", 2)
)

df = df.withColumn(
    "Distance_Maison_num",
    when(df.Distance_Maison == "Proche", 0)
    .when(df.Distance_Maison == "Moderee", 1)
    .when(df.Distance_Maison == "Loin", 2)
)

df = df.withColumn(
    "Niveau_Education_Parents_num",
    when(df.Niveau_Education_Parents == "Lycee", 0)
    .when(df.Niveau_Education_Parents == "Licence", 1)
    .when(df.Niveau_Education_Parents == "Master", 2)
)

df = df.withColumn(
    "Activites_Extrascolaires_num",
    when(df.Activites_Extrascolaires == "Oui", 1)
    .when(df.Activites_Extrascolaires == "Non", 0)
)

df = df.withColumn(
    "Troubles_Apprentissage_num",
    when(df.Troubles_Apprentissage == "Oui", 1)
    .when(df.Troubles_Apprentissage == "Non", 0)
)

df = df.withColumn(
    "Acces_Internet_num",
    when(df.Acces_Internet == "Oui", 1)
    .when(df.Acces_Internet == "Non", 0)
)

df = df.withColumn(
    "Type_Ecole_num",
    when(df.Type_Ecole == "Privee", 1)
    .when(df.Type_Ecole == "Publique", 0)
)

df = df.withColumn(
    "Genre_num",
    when(df.Genre == "Femme", 1)
    .when(df.Genre == "Homme", 0)
)

df = df.dropDuplicates()

## 3. Identification des features importantes

In [ ]:
### 3.1 Clear des lignes contenant des NULL

In [3]:
for i in df.columns:
    i, ":", df.filter(col(i).isNull()).count()

26/07/17 14:36:38 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


## 3.2 Scoring d'influence des features

In [7]:
colonnes_num = [
    "Heures_Etudiees"
    ,"Presence"
    ,"Heures_Sommeil"
    ,"Scores_Precedents"
    ,"Sessions_Tutorat"
    ,"Activite_Physique"
    ,"Implication_Parentale_num"
    ,"Influence_Entourage_num"
    ,"Acces_aux_Ressources_num"
    ,"Niveau_Motivation_num"
    ,"Qualite_Enseignant_num"
    ,"Revenu_Famille_num"
    ,"Distance_Maison_num"
    ,"Niveau_Education_Parents_num"
    ,"Activites_Extrascolaires_num",
    "Troubles_Apprentissage_num"
    ,"Acces_Internet_num"
    ,"Type_Ecole_num"
    ,"Genre_num"
]

correlations = {}

for c in colonnes_num:
    correlations[c] = df.corr(c, "Score_Examen")

for c, corr in correlations.items():
    print(f"{c} : {round(corr, 4)}")

Heures_Etudiees : 0.4451
Presence : 0.5803
Heures_Sommeil : -0.0172
Scores_Precedents : 0.1743
Sessions_Tutorat : 0.1568
Activite_Physique : 0.0251
Implication_Parentale_num : 0.156
Influence_Entourage_num : 0.0991
Acces_aux_Ressources_num : 0.1679
Niveau_Motivation_num : 0.0885
Qualite_Enseignant_num : 0.0751
Revenu_Famille_num : 0.0946
Distance_Maison_num : -0.0881
Niveau_Education_Parents_num : 0.1053
Activites_Extrascolaires_num : 0.0631
Troubles_Apprentissage_num : -0.0839
Acces_Internet_num : 0.0511
Type_Ecole_num : 0.0109
Genre_num : 0.0049


### Recuperation des features les plus impactantes

In [14]:
sorted_cor = sorted(correlations.items(), key=lambda x: abs(x[1]), reverse=True)

top5 = sorted_cor[:5]

print(top5)

[('Presence', 0.5802585402382587), ('Heures_Etudiees', 0.4451041402651174), ('Scores_Precedents', 0.174283049003179), ('Acces_aux_Ressources_num', 0.16785600902892453), ('Sessions_Tutorat', 0.15682926320393245)]


In [ ]:
# 3.3 Prediction de score a l'exam en fonction des features importantes